# GAVE2 CMRRWNet V5: Six-Hour L4 Run

This notebook creates a clean, native-resolution two-fold CMRRWNet submission within a strict six-hour Colab allocation. Task 2 receives more training because it also drives Task 3. Run all cells in order on an L4 runtime; V3 and V4 checkpoints are never reused.

In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import math
import shutil
import subprocess
import sys
import threading
import time
import zipfile
import zlib

TEAM_ID = "梯度不下降队"
RUN_NAME = "gave2_cmrrwnet_v5_6hour"
EXPECTED_ARCHIVE_SHA256 = "7ca8066fd93d590dc606134046dd662edd522d8b4c0c0aa40545393371cc1cc1"
COMPATIBLE_ARCHIVE_SHA256 = set(["7ca8066fd93d590dc606134046dd662edd522d8b4c0c0aa40545393371cc1cc1", "d72047b09cdb4ce3d0d74c1c5971a79430ce8354db9890f5ac5f10e4ea442b00"])
DRIVE_BASE = Path("/content/drive/MyDrive/MICCAI2026")
ARCHIVE_PATH = DRIVE_BASE / "miccai_cmrrwnet_v2.zip"
WORK_ROOT = Path("/content/MICCAI2026")
DATA_ROOT = WORK_ROOT / "GAVE2_preliminary"
RUN_DIR = DRIVE_BASE / "runs" / RUN_NAME
OOF_ROOT = DRIVE_BASE / "oof_cmrrwnet_v5_6hour"
OUTPUT_ROOT = DRIVE_BASE / "submissions_cmrrwnet_v5_6hour"
FINAL_ZIP = DRIVE_BASE / f"{TEAM_ID}_cmrrwnet_v5_6hour.zip"
FOLD_MANIFEST = RUN_DIR / "fold_manifest.json"
BASE_CHANNELS = {"task1": 16, "task2": 12}
NUM_REFINEMENTS = 2
N_FOLDS = 2
TASK2_MAX_EPOCHS = 40
TASK1_MAX_EPOCHS = 25
LEARNING_GATE_EPOCHS = 30
PREFERRED_BATCH_SIZES = (6, 4, 2)
LEARNING_RATE = 3e-4
HARD_BUDGET_SECONDS = 6 * 60 * 60
FINALIZATION_RESERVE_SECONDS = 75 * 60
TASK2_TRAINING_MINUTES = 150
TASK1_TRAINING_MINUTES = 90
AUTO_DISCONNECT = True
PACKAGING_SUCCEEDED = False
ZIP_READBACK_SUCCEEDED = False
SEED = 77

if "_NOTEBOOK_STARTED_AT" not in globals():
    _NOTEBOOK_STARTED_AT = time.monotonic()
BUDGET_DEADLINE = _NOTEBOOK_STARTED_AT + HARD_BUDGET_SECONDS

def _budget_watchdog():
    time.sleep(max(0.0, BUDGET_DEADLINE - time.monotonic()))
    print("Six-hour runtime budget reached. Releasing the Colab runtime.", flush=True)
    from google.colab import runtime

    runtime.unassign()

if not globals().get("_BUDGET_WATCHDOG_STARTED", False):
    _BUDGET_WATCHDOG_STARTED = True
    threading.Thread(target=_budget_watchdog, daemon=True).start()

def bounded_training_minutes(requested_minutes):
    available = (BUDGET_DEADLINE - time.monotonic() - FINALIZATION_RESERVE_SECONDS) / 60.0
    if available <= 0:
        raise RuntimeError("Training reserve exhausted; preserving time for submission generation")
    return max(1.0, min(float(requested_minutes), available))

def run_module(module, *arguments):
    command = [sys.executable, "-m", module, *[str(value) for value in arguments]]
    print("RUN:", " ".join(command))
    return subprocess.run(command, cwd=WORK_ROOT, check=True)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
assert DRIVE_BASE.is_dir(), f"Drive folder is missing: {DRIVE_BASE}"
assert ARCHIVE_PATH.is_file(), f"Upload miccai_cmrrwnet_v2.zip to {DRIVE_BASE}"
print("Drive mounted and archive found.")

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

actual_sha256 = sha256_file(ARCHIVE_PATH)
assert actual_sha256 in COMPATIBLE_ARCHIVE_SHA256, (actual_sha256, COMPATIBLE_ARCHIVE_SHA256)
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    assert archive.testzip() is None
    names = archive.namelist()
    assert all(not Path(name).is_absolute() and ".." not in Path(name).parts for name in names)
    assert any(name.startswith("GAVE2_preliminary/") for name in names)
    assert any(name.startswith("experiments/gave2_ensemble/") for name in names)
if WORK_ROOT.resolve() != Path("/content/MICCAI2026"):
    raise RuntimeError(f"Refusing to replace unexpected path: {WORK_ROOT}")
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    archive.extractall(WORK_ROOT)
assert DATA_ROOT.is_dir()
print({"archive_sha256": actual_sha256, "members": len(names), "work_root": str(WORK_ROOT)})

In [ ]:
EMBEDDED_SOURCE_PATCHES = {"experiments/gave2_ensemble/cmrrwnet_v2.py": "eNqVWEtv4zYQvvtXcNVDZcBREye9GHWBNkjay2YXwT4ORiAwEm2zkUiVpOwEaf57Z0i9SFtJNkAsi5wZzsw3L3qtZEnSdF2bWrE0JbyspDKECiENNVwKPZk0a+5R8PukNryYrJGzomYLKy3bZ3jt6I1U2dZ7SYQgVBMhHLNbQ2E6ybYse6gkF+05pF+ZTCY5WxP4p3VhUrle84zTItWyVhmLp+Tkd3vyYkLgTzEwRZDYvuAfbsVgIi/AwGmimJbFDtiSiiomjF7N7zraX0j0IOS+YPmGpfdUs2i45Q7U3tqGm2197y19ZmIzPz29SC8/3t5+v2HG2zWKcuGtlDJn4ILqya1OG3sLSfPeWCCqCxY7FRbWKPIf0UbB540UjCztY+qc4MhShAc2rAfc0pTwdbNLOEAhjeMGBdi4h61MYETygeiEa+vWuDnVup9ykHQNqzfSXMta5FdKSRWvo0+NVNK6ZaCHYv/WXLF8QZ4H8l8id7KuWAZm+AGY4GqKgeSgLWRmAzaONnTH5mlWKrUXzKS7eWdQNBuq35ll5YMW1hNS2fcE3c9UuxwaeFsLw0vW2nYp6yK37skUo4ZBArVxjMI4HG+VI2sQf8xEB++hkW7dmYmSYvxwLEY99VoNVE7YIzimiRf3cAzsMWOVIVf2gcpALsLaG6ZdU/BuDsnaGiTHcPTsAiRBNphHbK7Dd3sM2i9oCaBDikZfgfuj1RCQiW7YfrAwiKkm8rZUU2NUY9PMyhlQjVrw6V0Ko3g4hjyj2BaWppi4E5u8zLa0KCDFWSoVuNthA2WKKUGLuP0C8QilQS+aKveFCS2VLVXDBad9FEEAiR0bOreVQ1ZUwbenGdkxLvBTa1bcIR6DHVxzFHcJSJu0kR1ok4icl+TDklxgmIebeksrtjq7Q4LzMCq+0aLuPHr1COFmICr+vPz7O3HsZA+VkJitYgxdJASUlBnZAG7Ppq4gFI8eNw08HRCtFjOyOp2R+Yyc3c0IvC3uAIasoFqTS6mUVeOWZbXSfMdagGMhEhdG087Dh1HgUG0Uz1phrfJEs5JCHGUakjkn91jKYFexNReshMaROC9/2YK9suKwhw2jgN2fsaA1KsFeziBXoSpogm4yW67yHt72NKpdUoO0NX8EYQ5U5IRQ328hCe3mX398uyJNbUK9dH1fcq0xnWvtSDbAIqy0VjhmHe4MZCagONS2vaJVBaIeGKu0pRlqXla1mwJcPAlrAKpuXdHZYPPAasMeK6nBpQyC+Ql0A9E7iJ6cVFDceWarDu90c2nk2JMWJ+dVTLQUsoqbNO1bOSi/nnVvhuqHBbbAfgkRSNvoW6CG/Z6oy7RHz+1CwZ33FBQU3Fl703764GKzAPRlAbRfVM168qBNjjdlx2KT3+8l1kEwiXSWTj3jgB0fUNf3SDash3YbiyKA8hzh2xmWUPwyj16OFUUvgb8KOBqrOVZ2FPWMnx9Um43NKZ47yW/k4g3Bkc9Q1tqQe2iGhhSMwvcLX3yACBxw+tYBIUt7hJDiRLANgLeD1jHppHR99eg0FSA4UE70VkAx7Dy+XJLG2W5i+rXjWHOlTZpZeic96ZvZKH9D6XU+L9oTJ7cGqd0J8UC7GTmf+TBNfX7nLHVMq/jcFtbXuAdR6G/4QC8xl+LXBIW4OY5gNeAZS0dgxnyMx/YDMQHIwA0JGtuZOMQfcQrJ3z8lTwelS9WiG9mxaDXeX5CuM0G7xrh+14CAfz+hCidY2uEGIYb3I3SJq9VQeeFSA/MLFGZoAzTn1tlQjQW0kKYpOGFY6nlJNziJASdWdtCT5JI5e5uJvJeS2E5ng5B8PcH+yXUgr6Bqw2C7Bwb0gkpTMoWDN5zR5WuvvZ0tZTKsCy708J6Eptku91o8BDXDDRI9RTcvWn/PsE2mnRuX1xQw7SPGG/hiyzGAFfronqq8gdS67wh+BddmNVy9WwRlouucS2dZGC5t1jdneEXTlzA2tI2P8xZAbRB5ZywA8Dwi9KWb5CBKtGE0h/RwM96wyLqxohnZukrVSsMZbr447y/Z/SyAxKvX52lf1vRugFQzp6QugJcNEJpvSsnzkLHXFoehFPsm4L9p/B0WIt+Rbs5OcbzujTwOXFNuZ6F6U08gwHgo04Ny7mswAucBjbUvuu0sOQD6lWNHwN7L6OCYwJo+mh0EcNeN48OTZn6gTGcEbiPLM1/YIDgSHE1FHr/vwjWdBmXgeHhY3fxIOdQUVPNJfMUPNW/KxkD5vmhANaY5NdQGiC0QSLPCuZXI+3/g0jGoD42gZ88W9wtRam89ERScp8pFGw6OeFtN05lPj/0aCLsWHmx7jbql8xYDhiA9WpZgOWAaK9ct99h+ICZosy13sNwzvbTXdPtDzPBnIJcuwa1h7Mbw+m3hB24KP3BLsMExfqv1ft985fLr3SGWPv6eucsRzAPbl6M4j7lh+Ta2gVuWR/GcTv4H6RVegQ==", "experiments/gave2_ensemble/losses.py": "eNrVWktv4zgSvvtXCDkMJEdW2w7SAxhIAz2NyV52ZoDsoPcQBARt0TYRmdJKlBM39sdv8U3q4Tyme4DtQ8emqopVxXrxk7d1eYgQ2ra8rQlCET1UZc0jzFjJMaclayYTvcbLerMPvmSMRbiJGOuuZtuWbQQ3LgTB7WSyFftkvMaUUbZDLadFg45Ls19NNm3d0CNBDcc7gp4I3e057D3JyTZCBW44qmqSUyk1dh+bZDWJ4B/dRrShDLjZhvjP0yguaMPTiLdVQRJNLv7VBGxmkUd7P1s8TIYfaU0OuHkkOVpvQEXK96god5Q3sfqz0vb/SVhT1rAjrneEd1eFiHAtiWafggWloyC8im7k34w8V5jlCDexkppIkqJsGqC4zdaU4fqENjUsIMJ4XVanAQ2NTilYmLfSspsLVjJyoeTlhMExqS2vsqY9xEm2KfChQgfK4kU2T3z3xHL7qaJOFHn0QQnR/mrKLUfgQ4IEbaCRNmDAcfLJoPOsX4bWSQWCtkWJOViwILOP6WTEs1VdroFGPWjo7lDSXHso0fZ4OghK9cF7lNOD8Hw8T6NlGl0pv1DGSd0Q6VfxUO4z1dzKQYIv9LUgco+iS03epQYfCpHLbA4Sg40uheXC8bESqb77BwUHF82kiOxAMIsTfTrbcoMLxI8g6/H0N5zQFhcNQVXZUC5SXSW5O7N5djX3CRnZ4RHCnzXhDh8OOHhw/Z5gwGtaUH76ETHB69aZbILC7DcWG6Gj+lyxOlHNrHUYFmKcKIRoNk9WMqqCjgrBFprgws1W0oDArl4OnzfsGK73GDrnbhnMuscAqshvQbirM6zKJ+MmZUqqoiUJc+BImoYUaAM1HtoEYRuVCCMl/dXF+1yJkQS4hhQ+6fy/X6XRfKWbj9LIe7JYLc0Tyrz15epKrZdgILICb7OaFG2sv8+0vMRRajGaTn4LqUx9j33Bl47ZRVzgSuR1x22B+Zva4iMh1as8C1pC4SM5WCB4ok+Q9dd2BmA8NgS6f+WE481eNLKqjZMkuoEy0RsClKa6hU1B4tx3xKv7673Z+8GY6VaEl6CXQsf8Y7ulG4qLX778evVPEWswMP1W5i2MJ0qxi4uLL7/d3f37d6gvawwSKCORoFYNf1vW0eev8AWvSdFkE8nzZQ8zG4RNWeekBj9EfE+bqMKbR5inYDaKPn/4+ueHr2qDOThPHStEF3zWIcdrQiCs5AJlmST9fFjTXVu2jSaaCaKoos+wNcgA0TtWwjQhtVIyP8iQatqK1EcYyqBFrVuuzg7GviJqG03uS/TIRZrxzR7mRDCCRKV2l3VFZpw00Y1UdLL6Cde5OPttGv2Qccw0toVN67psWR6LNZm/aQT/rx6STLadOLFcO46wbRg+qU9x5AHJYogkoFh2KcS0KjfSsTi0kaSRWzmixTBRQGM3s0Qw4JOaY1lJYqX/TCkp/2B/cJybwVH8e2TlE0NYyI+VP2dOWI/LsulRt1djlNWp3DO1wpMun/xzOcJ+lOzHd7Nzxc9VNC2SfnVpml72392N5f6duQwB/xbi/QAFR2kgM2d9CtOiVyqyIDUQgisXR0jmhozt32HgdzEtEw/Ko6Vz6guOTEiFwkchs9VY2ytfyVgeBtew75B+UODhWvrmi553EgMWxQNaKr3CKEAi8V/gv58/jIkA3QsS3lxFK1oMqqn2C4MfLV+x/+Kt+y/P7A8Brza2JKJoU1NfRZMhrD2QGvPgJO6XK1Bj2TkBa4P+cBlR6LRDBnXvqsoQV0CEDNKxG+quJYB+dMAF/QZ9UFwJrmGbRzE0P8Kew8np2QqDreO3SfuPz19/Xf6L7EQqSlhkMHeDdAuyKLXfJHTQuc/AmOoI5G35LIU/q/avRovr1DPR1A+Ukw0+Dd2U3lMSrIYgyX0JiTw7gMr7FpL1jQHq/mLI1LULWLpLXgGEcrEryI+eDXSn6Dpoeh4zOldxXN/pOXR6Dll5s+SBM5i+cDHqSg063f9VNzDh8co2QITz7t2wpJHKcPF1dbKjlJaU4aoiTDmtH+hTqGO9Oj6DUjqD0pb0Ci4JxRlbByPEcfOS48Jlo7gVaeV6B+6eQcjY24laSdUCmP+NVkZCqvVKBITgb6QvkQ6coKRBAjY2Ye2ZHMZI9N9InPW9v/YwEjgBRGDu5WfxZXO7VVpCIo/k83dHyzq4SU/qiIUuILsCMl7GOTlCzbjR9131LY1yfqrcoviSZEdKnmK4G8wW8oqgG+crQGkPtpoZZaYOLrvNYJ8hRMRDqmY9gCvgmvlsFqzwECojKumj02egbXXO5HlTtPJVBD4i2f0pzLuy6Z9Dhv5iQbPojXdLXC0ePHRDozbeBXB15T+3etv7qVBUALzPZR2b276Cb35SVy/FnUy8arouyyK2ojLMTvHAO5MRuAQcpkueUQH8FsfefXQlBlTvWgkmJKkATm90fClMwyAQRcl2cE7Nf1pCvpFY03gIkNO0Q2ORmwCyia2GGUw4hxbqsQZrRYD76I3SI0RvPLhwT6Gag4HfDyy0Ne/1qKFlGYcPA5IhHDEgsIDikZaFevs3jidCmr2MIHqCfJDaJOK5PERywoFSWxAuS7IFrbt+FvcGrPsCZaJjXY042r6SInWZk/iIi5Z40b0va/qthAlfuG92mx3wM6ogIZZQciQtTBUyUtKo4TXNyQ0UxgrnOXTVG1Er514TPZKai/QbF7UQLwkGRc3ToJsHTQvcRA/gNqdsavfybuLSzBJaf89KabtInp4nuhsGais2mSp9jRd6Z3NcLmz8FxczTyv/JYTaWOoQhOVpYoYpJKaIGrMdid1pezYZ3nGTclJw7KHekmHWc5N303Gm2I825JW0mXsyVRsE0W8e+u9CN4Wd2f9iI3lL3L/8jsvUh+Bd104inIjXLd+H6CRQ+oQSBvMcNp67vtY33lF6M9IZQb5CZyWBQhsFPct3Z75208AsV48Cqn5hIrOPpvswOWcoX8ZdnafB6zUrvUN2Rr6KEPeu15ky9feWb+Dcs8vg2aDg4GWw2sXiG7/gQtygcgs9vhvf6A6d55GPeba8fgH6CBEN87K6T/XRR0jCse28SNfL+3Rzf+uyKiFZBsneDaXUZCdu1zVat9stUF50/XeR6nzFDeIypeOei/X8ruikVlfL5Bxmo95QuKXkHHSjiL21DnV4JJYhXO7wDB6QZR182pHQPTXL3H3Q1TY8RaduuJ74+JH4SdKPho/WMudfumQOXdxTZVg3KpIgrUxvfA9i5H4K8MIPRs6LCSB4H8oSpg9iXFLxy8Eomxqt/i60SQaBBM4gX0P45dW/PYvgSkGie49iCEca+S2chH2UFkkPzQtgGA31yKgdPJIORmNAqgCl6e20pUwOs+qB/ancGKQ4mODTl2/Xcps3YJe9OjAduaNpwb23IoNV4ZP/G4GRnbs80/6Ed9YYHzL9H58pWMI=", "experiments/gave2_ensemble/predict_v2.py": "eNrVW22P2zYS/u5fwdN9kVJb3d2gxcGACgRpexfctQ1yufaDsRC0Mm2z0VtFaXfd3P73mxm+iJRk727aAr2iTSyKnBkOZ555obpr65Kl6a7v+panKRNlU7cdy6qq7rJO1JVcLMxYu2+yVnLzvM/Nr59lXZnf8tB3oljskG5eVx2/7wpxY+hWfVHoUTWlybqD8/4tPFqGVV82R5ZJVjULNTveZl1m5v791Y/fXH0NA5J3S1YI2aU5/E7FVi5ZV7f5Ic3rosg6rhcL4LtvRXdMb68MkZy3ndgJvk3zA88/NDVMguVFnW3TXV1s0zKrxI5L4HCbFQL4c39c05b9TSmkBIUZyjK75WnT1jfZjSiQaVPtF4vFlu8YjP7M8855K7gM3bn54W4N246rbda22XHJ2lqk/ljEVl85j+sFg3+CIHjVNMWR5YesKHi15+wgeJuBMkCT1Za9++ENHovs2gx3yu5Ed6j7jnWHlssDbExU+xioLIicIxJLkFkmidlY1iXbdseGJzBjB6rrXl5FtB6k9tepbZyaLnYuR9iZKNlfEvaS1a33Qh6yhm8uruml2jhxy4Tk7Mes6Pk3bVu34S745r4BRfMte3n/j/ufmKfwJdvXHfs4IfwQWGlAXCVFglKg/nDEsofRS4c9bRb+hFcuAZqOok63cLm+Pis+npZa/tFSemDbmoNPgOxl1uUH75D05FlOzr5wMShfyJ2oRMfd44xiMJwwOiNW8NbVIjk52BLQrFaKHHpKzyWwIyJgXg1YGNlBXojG5bZkF/HFkl3GF1Gc180xjJwlm8trtarM7kXZl6EdXs6MXsCo/nl1Hbl02IuEoeWxr4DbF9Hm+7riS7aGf9U5tRzAr1JmirsR+77utcEqCnMGu1jkRSYl+1rID4NGjq/yvC97QJ26HVzSjHFWAaje8hU4W130iK8M0cQ3TOuUIBhoFjwS3JOXDJT87tV3g3cilKQpqjxNQ8mLHcJE3a0JRdl/GTj5khlIXBNCbmAM9EQ2sWZd3xR8AziwZOaPa4IV1NBgAUg6RspwGkg6xN/R9HVcftiKNoQgwQFakvdtD2rm9wjM9Qd6HC1S1pooOULgHpLlRKCTVhkR7plmDSvBfAHYwoFAZDFiGJuCw2MAkWmIyIbzU5w1TAy0jR/ZbRgVw05Qx6F5npG5C70VEclut2OHHxE7cIyMDpi9+Vqyspcdu+Gsr8QvPZ8IWfeI9wn7qLms2QXpWT+Spl0hHuz6c7N8STGaAwuakeKD0UXkTSPXUq4NsT8G+oBkcd3wKi15WWYAEbB2ycp6y5Pg7rNg6n/ahhPHDKYsNmsEEECY6Sug08tD6C/a8kK9dfyLdqGcyyoOnIjcBJ1h7R6y0RLiq9WUUjwGjwOEwa5rNblAvQlmT/uf/KjOeqJAjVaDU37OdoE51Ie4ao4OPGTb7YzwSzdmTHIK3/nP7up5kiuPPpVHnEoKtBC0eIilw7k/6uJWN2sAI+3trjtrB3cYuB4+Ey0VSI3j5OPcmyfHzac7yoyrGb9pP5tQ+ixRPBbnfWHwgxkI2Wg+10jtcjC1kmfVrK0Zpae0fo1RZjZ3/Z2tLcu7PisMGo1kdxnqiWBUI0ln2L2DcVGeNy2fygMcO9+KnGqp5RBecKbi/HDiwKEEOXG8cPCpPuPABQbZF53vX4oeIAR6VOhLFp3zOCcnUok7EZ/DYUzbdHBX1U1d71QhVt9VvJUHyPhMsbRmqIkNmUV9gzWQyjeGUUxBlOLt6vXoNcYwFZwwdcKQZsjHewiwAY36ua6QAkqerMp5SG9VuXg2yf0W0zJDGJEbCKlcDZdq8hgZaUyoPM4xUSoSRbXl9yAgpjY4sCHhguvoVGzF12oXutoEq4Hot7keAY3jJ7BqUJU36QQi/fDDtyppsNYLWUNRV3sJVTNgR8sh3cyAasVJnsAPkpYZnoi1SjyVYcu0wHpEwryp4z37p2cSC7VrW0hJKKn5NrTMKXHSg4bR2fPEXdvVQwmV17e8ZRz+OA6nTdrh9+CeUEnXYDVaBdopLBlt8Kmq+FPqOZiQ0LUurKmWAE0Yjh4bBzQUY8tEes0N7Gv8CxCAt4uxT9KSpTtDaTvnTcfe0HLaM3ZOYHTN2F8Bg7J9ma3RhtWOV+x1XWQ3gNwQTLa8yo+LM0gXvP7u3bufvucdu71y8Awk+qUXAA3s7fE9ShVEalfA1uiGOilDbyXU0iOkqUIFMIXfipyrtGpGee62FSrq7BBQsKhz8pFE0ViyOy72h06mdVUck2+zQuq8UKvnPaAXbemUVsCE4W+9nU8RIfI3bttLiNdFOCgidTXQZfKDjpZTZagGU162Ldhd5/auWo7tKOeNshWt4hQcb2Sa9BoSj53YpzpXH0kUY+2ZVlnJwRVpYozdvcBdCqtwjBQhQ4dcDAJtU2zuhWBRNfaSkqDvdqu/BbogH7ihcCdsYyQSZspNPwD68FojBgkQECKo3+vF2aj92hL4XG+oFJJ6KWvARJ+50zTR6iCeeGCKI/7C2tN9S2edUntATQpe121LGPWO530rxS03HhU8Iux/KoujjvIUt74lyzstNQmChzyxlNByxQ0k+MfSDt0gBucQAypeyARPRvHbBN4biGPDmqov05ZDIstLqv7dVaN33joAWXFL23AsAQ2HnFfNi+KuDo172X2R/aUQ1mFnCEiOc22CYTjAfkfXwi+nBaEIcAiyoQftNL7U6jWenPUd+Li09jl1UbAONRgD37aT6ERgmP02c6s7D0UsUbUwpbxKLTFZlpp4Q5nW5ZeenE4fPbSIo4GZkq9Q70RmZVMALlZYOhTiV20wQ0ZFXSFicX3tbQ1QqcuGWnecrStUKmopAf41IHkFToozwMX3opNPwqUb6mYmfus+3Kgd6JRJlNkeS0iauwnoEexpsA/caZXeACh/QCNyThz91C7Epyeug7LGCAVpNaBbBWlH6OkT6MEsNDR1cCRWTL+NSk10iG8Fvwsvl2wF/+G/iovsto8zgUm/hYfRXRiqXyvaWwTVARCOAa1KqCpEFV7y1ZdRxF6QynQ4ziqJxScm25tQeSZT0dVmaGAt2BAPdoVopINqw+IYrBWSjXATqtagIrBkhiApHR7VW3rSx+5XziCD4opuppUmqh1vIehwCrRuXU45OsiUgunR33eYdA5S+Tmz6VGQisYptyIzTbLNIiUKzlI9AjgYUcokXF0to2iW2N0nELscEyM1nECpaMrAyd+S0y6rAET3OiJV7oVP3oTHw9nJMH56O2cU/QyyE5V7O42zBvNeZ6GBV11DK+KA5vmH0FsZxdToAC7JRbS5uI4hNwmjmO4q4W84APDIcHIRpmB7/uLP1NYK6WJ0OyBsYT0vgGGKERriGWYRdXsMh+xx1LPDOzRMxajvLl1HUDezcVt2LedEQDsXTj/butd5tJan5SCQ5Dqj1XHGjYm+QI+gvorEgMNOkodJC43T0z6PMRjAxsPoGdFWnSCOxrxs8J4ygwTBRsu2r9Lh9MOs3cu1veKOv4cEWDYZ+I/f531kM0oseInkYvXki0wQia7qIKRdRXtYXwQEp4P4QqbZbSagXCtAfgaZF1cp8SilRmScvc+m3cW4YzhkVW0kNERp39D2ohaGeuOPmRCq6+Nk5n5c8fCGIhM8ROVcj9hLerUCi94U+9hg/w04RBLQAgjC2hjmr93D4V7eMoAY/tgWhg5Acq5PpYsV4HyjUmyYjhZtzpLou6+F6ib4rXN/vVMy0SXamEb0ePVkOz920bgasaeKJQneK03nmot+3WGJnnZP9qaic/D2hJH1o+X4sNYPzpQHcwdMR4sWbg/XqhEdnbwCziPQxu00v/RJ2Bu2wZzOEHiSnTlMtKUixGLpi4K+h4fLQUrSKTLp1LiSEyddBc5VMzqYuSWldXqYbkwh0wqcEiyAZ0WbZyX2wj4fJCCS88g/MNK6ndXDKP+xnblJdY3JEIfoxVvMth3QGGUOXmU0FO/n+xtL5sCMW8D5XVI6WNObH7cH7dnDPofeo72ESNy+47V/nae+DQLa7qdC4SSrGNnH5L2yF/pz+nIEppP3RvzEbHU6RXfR4KBbcFG38B3lPpA25FzKxNTVwxDUBd6CsZZbnVygQrVa5rIr92ufs+WkrezdCiWwBaQ59y6LJlx8AJtHTk94KjS9D46G5U43LMPvnlILb+6sxSxhf7vzqdnsQrKZ09z8S03nxcYF3+tonPDN8prqb+67Mp+lA0V0N6zZ2CvixsZWXx10S0GNcrrT+Dj4FWL7iMqIwrmsUHdgwaodVBItQLQ2ynMOOg/eyhknmYx1NoMPEvQKVjnnYZZL5Ore+fxjguWjd5HFb1Mn2wtFb95MKj5Nx8e05wQCYU5/bDSmMCgDKuqXS/bC1Xhct2IvqqxIpfiVR9H/R7D4c+G5Ve+fFs8di4jxQ5BZgPHdejk753cKBVPip1FoSM7+EDDS32umTzGpT0OgwfvOg89i/lAd4daL01HLPWL1AYZ/nKoGcPw0WnxqLP5D4vBoN789qC6eG+E9uuMOzB8Zdx/B/4X/DYu66aZWjex4SdZiEHngG++L+iYMXihWtlsyfOGi6cx8aDh7+fR26LfpT2nBRAQUp7ixjrtfvahiU5N9WIKAfaWHFfvoYXSLbqU2X8Vj5yVFXwqp6zJtxqx1vwoGW9VkURNetfseb5be0ptwy2XeioYuZYN/09UPcy+wEUpWvJK8vAFnG5pAcRA5DBAyURqiHAarFbrjCiEACmdqwau7W+3eW+fO4ASBtq9WsN1PXU5imw7FpxJRWv9N28BydSW2wbPWYIcC04kaIFgmoS6jl6qevgqi5xBDPPeIYcG79LoGz6JnjHi1020RdcGCX0SDXWZ90SVfnN9dl3nyVACgKJC6CokGMmbkrJlRmAqcRdQuPLvGQY+phRk6lMgG8WDvqbPMoMUJBg7gusT95rZa6rqx9uwyE1U4agvb/ytg1Ih1l5vLH5USOGA0ANBHRYcaTQuB38Nj+yRNqRGSpsg5TXU3RImx+B/Csf0e", "experiments/gave2_ensemble/train_v2.py": "eNrNPGuT47hx3+dXIEylTO5KPM3M7Z4jW1d1Ts4pV10uV5ur8wfVFApDQSN6+ApJzaxus/893Y0HAZDUaNZO4iuXVwK6G41Gox/o1uzbumSc74/9sZWcs7xs6rZnoqrqXvR5XXVXV2asfWhE20nz/SEzn/7S1ZX5XIr+YD63otrVpfnW56W82uN6WV318mNf5PdmvepYFHpUgTRAxpn/CaleWeCyOTHRsaq5UtDpTvTCwP7bd798f/OvMNDJfgFrlc2xl7ytc54dYF+y4B1srVuwIu96ngEYz3fwta/b7MCzuihErxlNc2Dpoc37E3+6MfTjKwb/7etix0tR5XsJVLqDuHn3fkEzB9Ed+D4vpPMVt8NL2QtkVI0Xtdhxj4oafxJFDlByau4ZWIGtHCtnPNGs9q3Iq7x64Mc+LzqHX/EkW/EgedPKLO/gSDki8APsvn5oRQlbb+ou7/MnyZ9l/nDoOwWR1ceqh/O/2sk9exZFwfEIufx4EMeulzslCBBmC1+46NdsD5vSrJbiIyecMq9A/p2eZP/NfqwrLZqqfvaH2UbPJmz5Lbuv62JNgPl+RI/lHcEqAPyvlaDDFfujKEBH57B+v2ErB0XknWS/iOIov2/buo2jEUIJW2X30oooSgg7O7atrHpgGGWSljXcl7rKszjBZWFjhj8mgR21yRiGFbZm1RBZ6vlBlAn7dqMHQ44S9oa9X6UrfS5cnTIn7Y0Ttbe+PQ2bNNcPAewg6QwNpaQv3g3Cy/MDqKdsr0LpEsrChcBJ+TGTTc/+ROgkSbyeMLpm7B9ZA+pXijVIBW4jKCPsV1ZPeVtXJex+2TWgl/s8C07lAygfiFafy7/8+4cPf/5R9gz02mg68PRfx7yFQ/rp9DPyFSVqX7Cwlg6YAN5Jueti/P81g/tMqjVojrJRKU4TjDqgqkmnJ/T+ORx8IPlhOoXLeRQFD1BBL9R0dtyJNO+4eBJ5Ie4LaY5toEAgDhkOCqBJqY2JY1+D6QL7paxmrBnbyac8k2u4le2CibKhT4m9RWo6JUXrnvP+EEe4FAgOdovwbLNh0f3++n20njz61CwcK1K8PzVyo4jA6vRNAd6T+l6/9zTesfPxsJUH1APwQaDkmaieRBffiz47rNkuz8CEgyV/VPvAo8Ox9SVnkZdg9TqYJGLbSH2P7rR56h6dOfqqp0A2D7J3JvWAngYpatVQ/8CN/z1bpe/cC6cXVoLYF3kTqzGQUF52m3h5vUgSC294ccBpaBp6YM+B14NjjL8Jtzev4vbm1dwShlKTh3sAVctv1wu2vj0r9N8O23gAmwCoGuoI3rFuy3iV/vNqwa7T69XA0n0uujHkEkzq7YLh/ycO0bIUl1FVnMf4zxvFzFtaKUmzAu4V4BDKKkmb+jkmwmdP6MY5Iof2Wy1HhK14kT9KHEafMOJxRbu5TkYcXE0cewaeBiktHOHfru8SOqPNNXk1NZNCrNPI7fUd+5bdKu8GeNbCgSqQDcEPNxGZlantXTteWIdlwMvtrL6qhb4OtBa5NNhr++ktA+Y2jHwkHbh//62CuZPaAMAcfXKn7PVHYanPrkkjKG3KKpA8hG+/Sk6j9h7pC2KmKbBW1m1Lhhoj0S3Zy7u75CLjVkpR2cMTHe9l1YGf9FaAXQFUdGfssj4/+mI8hR2kb0n6lMvn+HrBwIIw/F+iY7zdy4sB0N9iLS3WWAsPggXcRcK+Qi6UKmMoFF/L5fsEFV8dmDoAE+1T7KwOi6J9kG+edfFOZQVrL0cgr9Ifm0Juwe1XO9G24rRgw+c7dSImAgRJwNyvsq27+NbsF0bo/N5/bQKBXhQXQcJtZR1sqpAQnTDD4XD36xzIKIAUt7pd4c2DK2FBLGNvLaDaOl4PwL9Lu2MZi495t7l2LDNx+NaEmQBHYIl3CucSg9hMLhQtsBV2CE5LjYG64K5js+vbm2TqqMJ1zpzUcC5KSBpDXZXLzt/bosbXXEFg2efqCtfHHmhxNOKyi8t6J4uFNgCQS+V1CznNwEwQVr5wfRW+0o8MXSL8KzoiFHtLhFoD8oObIpdfkymHy4GfdUpygCBRVpDpFfVD3mvq8DnWq33FYoWjvpsIAaKxtoI402KFhLZbcBw3sOCdcsYYN2pjUNUcUsidG8CSoNJ93kJwekxBiBDvPaUoxTSrmxOPLeQQ7g42JeDGbP8MUd/EnANUZscunwQst3KfV7L9a1nerm/ufLYnCU+xPQ04YlsnfiXkUtrPeOGylwJc5nL8uHrsL0O/FkAsAi8aLJvMUA1G0r7WSQXiV/y+qLNHSPM2P7dHmUw7a3/gcgquTw+HLqAy4frxUUY/3cCWjbkoKEeGIAUfbuDWVTMZWigxcyDqrOr7v8is125IvfYUddfhI4hK2Ju2vhf3eQG2S2obrZTx6qJYgpRPAvd6QBPfsK268F297zmpeQd8AH8XObaBqRMHz3KZ29Qm+0Lwg2h3r2eMsC5dwnom+17moiHezbv3ydkw4MtQ8cFSMYmBbGB382ovW1lB/o2H55pfDCdILzGaUOq39syXmtyM7YdRzcnba/4jHs4/PpBWJ/6iSh8kqjS+KWOYjUoXGpJkhISqiC7J3J/YobIYXd3QEPn0HIUEkrN3xl0iUWcSY5w58VTgZ7lTtuQidEyDMcFTbvbW5zpQ79jdxRu9ckKhG5Lxce/zSrSnEOtbVKl3dmtXobwhRBdNI6tdrCBwDNxQL9BipFlzhCDRxxobCIgr3e8BekpP+PHs8ZDiv3XP6DTs8BJajh15a7K211EYmxYgFGuBTsj9YpqaJ0XpdfjGJNigPXB/FL+HJxVSGHlNzBJGOYUxJSanBmMCCfmDjG8nbnYnC5CRhDB+5oadnET9bospyTwNe5eGHOY8JkjSecegpw79RgJpCj7ATjD3Bkzvu3QFxhfv3Du4DEVdPcRj+xMY8K1hhdRKvXPmFeVDqBuY6ucVxs79YYPG/cxhTjuXcwuM0M3uFyaJ2QRSDNgZEZjij8oXvTYE67OP8r/YeEd7Gix6yLLpT7pKgnYBrc8NpB1vpqwEoyweM5PR7ffusIKz3nugOXFNB5rDfZsjhk8Lna+qBPMVVo5ifd+cVzO4EvKB9Ao8Jwg7p5qV5mW8BjJhL+0cD3CvMLwEOi9W6uKxwixCFdWSJ0VAEl0GPFMUcYtM6kpUk+LWYzySJAF2Vuk3w6zmaBuNGDIuVQfAn6x6RKgv0Togr5QoSRYDHCkBuFc5Ala8TIGawi0uoHKW+AlrdgmZKPqIBgrh7xx8Qu1X70YroV54KxnICxdCfHchV3tIo86jE4iLj9U9R7MmpTOhed4WJmhcuJsJyi5zgSYBrWBkgH3zRmuOGvpsMlUqdWcHmT02NdxVEy5iUXxNpX0vK5qqYXolKCQXN4RWigaoZ2SDNib8NG9VdVWcNlQOTtwq5c8QbZMBo+KkLh2ye4ly8HC/ZHWTnVONkp6hYrj13dp2T6Q/ilJ2jQBQTPJw9+uhoWCoAE9Ulp30Lyvb9rkCYzKU+bNWYteAM4NS3Ytj0fN6v8+zHGuJ9bHN5Gwe+QdRCEgqdh9kdmw7sDM/wLQDbXsMMPBv68LgfS/a4vSffd00MOnlnEOheCb5VGJD4wdi0o8dfrWSygmYcjglSYtFlcf1KjpbXVXFgwh8XfRiITSsxN6L7BFC4Q7pVlV6D0nXoRTtI6yN7wFXupqoujFgcNzWQQqQekNq63hl8x26H68TRcHjUyVv6xp8eNeAZdlERvjat073icS2McSQh3SOQLoNXj0zvY30KFh06/cto1hzYVjEtwPfUjw/iX6mnWIf/akiPkm72SdL73OUWKWnmoshq4luLaSuydJ1UsLC0e0gjTsjDW96eIoBAO0RTSsA0caP4PfsOomzjn4mBmLu6/MQfwUHZMfDg1o4ZdDucUNo+GkYN2e+sRsc5sDpgmPJZNcpzOG7grFq8P/Br5b3a7n1Xhac9/pRd9boEgzMDMqgR4elgicbWxrwntRfLhN4WuAqRmEs2WDWBil7WINgKNniXf6rVIIZvjsHcTju94WkF8ZhFKJy/ly3jxDbKlT9xRE6LFjKsm5Pm1mD5hyeam3j+2rj9bqF6nR2l46+Te7xerwrcsHT28JCnrc1dpP8b21veOukB6XQW8YvKj9GRzamMofpDPlbVI/52NmiYYPBAVpAMPVEN8IJkvDBGfMwhVrzOSC9u+HR2lR1XlHH0vUj88YG4pkMBOJRxdHcsM3oyjletKmL+uGkZ7Rc/cGZPdQNRECwg9Zm9zSSfrcT5Z/VRlKIqSCewvQvhmS+aBX9ojVRIN/JTJz09XFGdJYEotwdi9EKRcvtVPoB/s3kDx/+o/oJ1UkcBzlYDhdeLWmDbYSONd3D4dXtZpW+c3QbzhIfcA3D3Aw4oju0sjuAiVIwEoMsMFMqysIaOGylMO2ktDaaKC0C9dm92g4+bdgL2uKzG/CZDRgZM25Zu4RxfTFV+HsQ2oOHrbWxHzplwCQ5BgDHEDrm1HHLeZKCzOriCaI+1A24aEqVbABMK9gW3XgmQjbvZrSOg+P18caWi2SmOzdwwTi1yx1l+eS9yESQbVOSvWa3C38Go5VozWzkEkxPSQwzSC3TAJrYNpsYoM1eA+hAMgO8K9MB5/P4VGHLJr7XEmBfAReD6Y3gu7W38HmvNuSGi/o0qn2ODyXOu8OkzKj7yMjLN+KKdF7t5EcDgCMurgkvYH4iLIt06AOzoyDo8oOIyHzxrBD0bEI9EjSUpJxXYNI4d4A9N2PYnvE9UeBnDPis+4nmXEu0Zpd7IP0QYiIxs2oYBioO3SiQVnG+u+QCrwKg4RD4DEye4sQ7fMe3oXDd71MI+u0quI14CjAKwle+kN+D9tO7yh5y/oL3qHqPpyi4LwilmANk7OML5omCA3AdAmi6Lsz7VXiFUW+waVqdxzlyh1y2AlybR3D1bnRjX2caXmsWzliwaIgeB9Uex8gRNpqAEmbH0oANI642l42Zh4/uKYMfoseLFry4ASla905C1FTCNZmEdB0qQbvhhAFyxwLWc+xxxl4fj3sccHkc4oCBwwkPG834YIP0oouO5pzyDIWJaCMKojhrb6eCu8+OAScvavy88QY0k+IviewbjZlP5Ue4sRDgqSb16hTbGYxY4V9w105D834wWxAMHEvpV6wmKhn76I/4EAGUJEZqJ6xjIA2qZfwOwp+uY8ulIsbwaZBeUPsDljs+gl3E7g+wH4Yt87rhcOPsHB+nKPZI8IkFd0xvil3swkB6Apkp1tbh3OodZgPRsd8vfxslCfuHjaZ3wca+R9lheYGeWrQH3dVSbbCkZgD8HYXEX/TM7AOfz9wOAy388hFFr6KsTuWujI6K149Os0xw8KkKk2hvtPndsWzM5hcMPXPVb7DNbbRzFVNQpaM9TWmRnnLUSI8AnCNol8J5ScPpecBWE+lF0bTHYFLKZVNTX8W1aqHB9ogJDnE8bfrIe3LTioXabfHsUoPg8YUC3zzn3tMt7kK9dYYNboRGRNCN0c+Neol9u2bEbZiwqc0cmgXg0wSGBGpuXQMwR8ATK+5TI9IQQGIVzUJ72U16rx5jCSEF5x9HOMLLGtLiunUkM0azK4bIatk5VICG/INgOpddRcCdjRZslXgGIqB0qI9UCqibddAggVS1rRIFKu5J4S4VLoyCzSL2PylZ/Ya+/ebu8+/Yo5RoyVkm2x4iBwDGXaEuBi0CqthhlNZ5ci+USIZfR2QpPrGAyYy97Vzwth780onsLM8EaERsM4dWb8W2HjgKoV/+tLhBD0YNqBS5x1P6jO1XqmsVn4T7mld1JQN7BXaQAgDdfmQ6sJRayobGvJYrHF0MjVcSom2J0UPsPhwulE5vrpPplqzZX0I5z5/J/3kzl46k/h47upSdgau1MydF/3wVhodXMwhU3XkW7S7ozQAlxgNl/xQSwloUVUZoGr4UsvKOeEJKulO60j+0xIiP1I/jyfDJdyw/PhzveVBm5CM+B3CJto803jYYTbV++cKk2/AW3Z77RGzaQjYzfbHDM/NEb+ygc7Paqw05M3wOK2xHBetJn0Ry01Qc49Vx7Tl8k3xssNJm4BfKMG3o/0eRXqlCvD2+wFqU5IJI7ce6WiosV4KqfWOw6/SPG2C29bP3IqKie/JVa4UU5pdaXSnn9Y5ddb3QkeLPcXy8N28GrgKKYb6kzmTQQVJujj0GTbdd3W0hzYFTCfNm10GvJxzzFLzd54wbD5/RXDe8PuPBnezYfjLBpe6NbM2PuQfX7ISzOOuEvzZW4yaI+zTizIRAa+3BnIgplNVk7LX2zMIs7mTYtfauxizuObX6Oz0/82RnqsFrNqvGKgWh5w7KRSZ0QNnyTjzJODxTJ/72bIK2KVOxzwwdN18Yx2hePjKdS2mQF5Kp1wegscvZgKPSrcRlZuSQHO4m2xg/TY7qQ6fYlp9Rvr9Wscb4bpvaS6o88ZDmN3CrQxjPJuOh8Jh8iCD4gwzg0T3Gqb/REbQVad86+qMS/kHLQjQdwJs/grFh8ejPXCzDjiXsdnzvRsoqm6AGoc3EAV96sF96oK+1SMrC+TtHNH9kAicUpn0xDMZ91M/zV4uOcOZmuT6GZHvhLR8lkMjZEldiBTiNHk4K0y9IIfcQjTHvhd+PPdbwwZfJOr3Zf2Zm93NKGiaWqlGOmuI4yiumhrhxn5z+9S0O6jqSAviufThiuvQTzcQ72UEk2VAvXvQzxjhgx9uW2qEZZlRLqgweKapy/pxIqhlWC6Rit0NuiHIcLZf4EL7EGh+k7fQ7HdWvqP/0yM6JpGcIQIS1hO1+KTqKa2kqSl9KhMph2Edfg0XrNjHVx65hRP9MP3ktR4YR0I/XoGLJamnrWC4NXYfdmL8XMkMA0uqlW9uaInHzAguQTi6p2DC5/llkzKOWqgTx+pV1i8sXYNp3owl+b1ZnUcEGLemmm9upiVCIfhYT6yiuymDeiBpDfxwmGTjQA2fvEKV1Zmn6SaPFpsbOs9hF6zHtCM3+9Hlu83k1jz004M+dF5Uvlqq+M0vj65cVhoo+MwTS1Qt7X9rSzvTxn1ec4XEQzPh5Urer15BC0ap60RdKxikVT2lZVogDfQAJnrj99gTwsuCyOmBJ1lNDwjxvBXVhSp/sDOerF46ECplTAvzmm/N+gB75AVNkykdhjA5OHWzmebarejmU3pd+hf4MNfMXHBRR181qz1vSA2nwZwuGtnUXA1EgvjSdCdSRzTkS4Fx3ZStqV/8DnEViNQ==", "experiments/gave2_ensemble/training_utils_v2.py": "eNqVV01zszYQvvtXqJygxcTORw9u3Jke3k5PPfbi8TAyCFt9QVBJJHE9+e/dlcSXATv1IQHt16PVs6slk2VB4jirdS1ZHBNeVKXUhApRaqp5KdRi4dZEXVRnQhUR1WKxSFlGqlJxzd9Y/M748aRVnIG3OClroZW/IPBrNELzpsFnbh8LLnhRFxuS5SXVZEueo5WT0I+h5HGFooAsf4XIkUiplPS8GbiPzRroggJV5sVvQ5NUnyu2BZHx+PNz0IGZMrQoZ6x4dhU0UidaMfLDtu/QLlqM+JOUK0b+onnNvklZSt9rnECmU2tJbN5IUStNThRE+sSIogX8QW9eCwCxim6Dbg+vW7IKSCkbcX9/IBtqB7ewfXtj8kySE5CA5T08A8yCHal5qfgHy5VD54gACR3EX16HJw9XK8ZaMmChwA0kOa985yxsyBI23AjgrPB0/OZ0nh4DR0nAKemRxZVkCVfAX8vJE1e6PEpaqC5t7Vrojq5dMFxLeaJ3SsuQlIe/WaL3Q8pNk63v9AbtJgh337JHvSvSuVc4+1ZBpLxA+eOtg/6jzYo95QNw7r0k7J+a5vnZ0i5tiLA8nJcHLojBrGbZSF4naAiLjTz4EqCmGKhkhIs3mvPURWz8IMe6dNSFD9RQ27VTMpwkbXquxGPUalQ+zkWvctT/KBrBWKpmCoZ90KLKGabQ+Evqos6NKB6SCwSIvFncbUKy2SzX+5D0N9M3lwxZC6wfODBJmLFu6wQspoA8oB9Xdv5kqJCs2XL9GLgKToA6s77aVCKaP0vB9j2rmItEsoIJbdGnPMt8K2owh4i3YiLF4viXyVL5TsOWwG4F21sHgQUz6gXWr8lpu/TjKPooQUA/JhLWI5zCvRiCGJ2cZ4h5HG+Qvc7TVcpM07u03PJGfjx3GWK/KxgV/kgjCMIb9rGjpQJHI2GkyxzKzu976KCOQneiYNqiH6xbnYoyxomJHEXExeC+XT8uvl9F/GwuCKWYUnHOqBRcHGMoSmZnFdOA5Rmtld6N2v9+MLjErCqTk9pAd8KjX78MpSOI3TizitarO8pmP53FOnppBqCZS0kyqmBUc9BBvgez3b5tdqVud9e1MMc7r/wOGfud5gp46TlPsLLzNHQ5zFFjS7girKj02dt/2tZTYiPTLHXZgKBAdh9S4svyPToy7XtG4oXQXAOSQXsFAeSscdn245Gr1+tM94AbiBGtsBP4mVcKuKwu1x4+ifPUCn4xbZlcho4/3cUCwFT8zvUpphVmD4GOASPWiQJFDVTERj/Ha6czOJV+0NkdNg57zbpgWvLE3o+1oG+U5/SQNzPiV47XPcHadeJAOMqlvakOTGkAGuNO7VH38YfkOztvc1ocUooba+oYHncTKdu7Lu2cgkOr3gsybdZZub6LQG6FsunfD/jXB96SsIHyeqOO52mI1mR8VBfndRM9Z59YQgeWA4bLfAhU9IagzE5f73WMOWztOv5mgZoYlzbcJnr6Gt5O22sDNaONbclGA7tD72xtd7jXyrv+MPAEqLB6cIIwJQfYHAPecCSzJ20e8awHtgGkcR09rm5UmyZwO0CK0HtZ66rW7Vx3gs/fmczZNKFrb+5iN6Voyt4VXycaF2Yn+0KB9rRtjidGCJf8u6rNLdwyYXiBglotFQ52SqOd+0rzzcRu7kNzU5mbyJyJu6NMkwcVzP+tQfq3fvrbYKQ328LpS/hG4bDSI4jxvd2Sp9EVt1tFj3ABRU/458WNnPTdfYRJKo7Mh9HSeviJrOc+wNDmYYv/zAw5OOXdPP1Af7/4D2jt00s=", "tests/gave2_ensemble/test_cmrrwnet_v2.py": "eNrNVltr5DYUfvevEH6SwKsmzi4sgYGW7JYWNskyJO3DMAjFPs6I2JJXkmcm/75Hvs4ts+lLqQmxdOZ8536xqmpjPVHtq1RPvPGqjLorabTyHpyPCmsqUku/Qo6el3zHaxRF8/v7BzJrb1SIQpUgBOMWnCnXQBmvpQXt3SJdRrf3Xx6/fRXff3v4AxEt8BcSw7YGq6rAFIf7s1xDKkA7qJ5KaElZZe1GgxfrlNevMWrNSukcubmdz/++A/9X+oBWOjrYy8P1Rjpg1xHBJ4eCBDriRWXyBm2ErUKEkDoXuQEntPFC5muwXjkQhSzLJ5m9UAdl0QsJT7hy1IxsD7YBuuMR7yRSxiZu09gM0NVdNgsyFx62noLOTK708yxufPHhc8xOqflT0/jGWAuZh3wOWWOdWsPgeJz0Sk5i70wLH1FzKJSGEOn3QjE8ZC3LBtDsH43CrIpnK/MdbAv+dQy8e1H1oy7BObpfUhxV58LVkNHYG5utYkaUIxh2cmc0JKSjDjSlncccQN5HZcxgtgpk/QzC2BwspnONRintwWpZCrygF4FLa3x7g/YC6MM89iXsXt0hqbUimqgFcd7SUKusNyygeOiFSVwbvJ7K0XQMIL1IJuQU4raRdiqe71c736n0waJDj4OI0eEdS3sKllvrA/co1Fi6wOeSXyyXCVks0uFwFQ7L5WTYqAXxZzXS4cCiU4Xz9UeDPKOExXVCLpZceagoSwgawt6JupxQV+9HpRMqDaj/ujz9xgg7tpkTtcVxk4HwKyzDo+KtLeQq88po9y8qFKo6jNlj+tZvrKz//xX99jybzNwovxpdxXkekNK+flEBaewrZUQ64qv6wOZh5Lb7CH9mYX/gzIfStavjmJlvLJZMN5L3fg7PEFSeQ47eHTOEJ47jk/S9BPyMgWsdPNI6OsnaLbxHjNFtu8Co1rw7seuTgKEsBXau8kK0BZZgtusmTNGEmMYPxydclXg4I6kNV4NJxo0+SmTn2UOjjkowJeP5rL2FsRtp897c7U9ssuAbq8k29P41jgwLNUhPL5MD9ThP8I+di+0dbHbCOx3PWFAj7t0VwZIj0uEXwD7HfnjbGsYovt08J6pXupdZHP6n8bH2PuvtpnSzj8cMuql2h9ksPWaROL3WMkwwlATZS21wPQSPfpelg2N+UxQqU7iou96bda9zfncpdOh5GwHaNQvOZBey/CkhnzGxKTu3jXDk0l5M2Cbszc85nM7UNzWmv2PnbiVrYGQ2I0HZ1aAsFGlvWBifg2y0IVKh47Ss8Cs4wGIhKqm0EHFXReMmClRsoH8Am0xl9w==", "tests/gave2_ensemble/test_prediction_v2.py": "eNqtVk1v2zgQvftXED5RgK3aStIWRrVA0e65wSLYi2EQtDSy2VCkSlJu3cX+952RrA/HTopFqotEzszjG87jUKqsrAvMH/1EtZ8ByqpQGrpxbVQI4MOkcLZklQx7rbbsZLzHYWvo3OLSZo+dGb2z/aRDMnVZHZn0zFSTyeSvL18eWNogcCFoRSGi2IG3+gA8iivpwAS/TjYTVTAfHKeIiBkbmDLEOCYyqwnDpxvFynhwgS9mQ0SEi2Vaes/uHeQqC8qav5MH5Op5z5qGn6SHqMXLoWA0L6o+RGRaCZllUAUvwt4BCPhRQRYgF4XVuecedHGKp6fZFnJxqqRM4p08QCIAGZZbDfEJWhySYbucByHdzk96FBwdcJvW/QQ90yE2ro7T2blxPs9lkHNnbZjO2JQGly6uNvNcOXLAT//mcHvpQ2nNS2lUgVsx8nxDBtEZ4q/emstgW4eqDj2Ldugv/QLIcq5y8qHPKw7SPzZWfCeX5tLm0Cxgi0tjV6EmFU9uNyOnTf/1XYV9q9bYbr9iCEdFoTftPkbRa1TZU1081mUoGY+GopESYlQcSvHPb7XUnBzic7nM2M0pohdbZo1XPoDJjig8SzxIeCUm7oWzKD+Tk+jw5cUBEF+T31ZulVbh+BsE2C46AlUw0uJoLUzdVPFPcNZzfjNjtzN2F81YHo4VpGgptJXhJomuxa4XGwxfxO+vGpetMblqTFrj296Iu3JG5Zc8MGC9XCHj5eqWwJbxYkiwFSmV9dpG8BGTGQG9VPFmYd4Crlczhh1ptYl9XfII+S3iRXQt9sHVwJG21LqLXRLVju8f6YnjejGajl6JlZxjnYsyV/6R+l5d1loG6wSKyMkdeFEbzNQE9ZMk3e7z7+qEn3HR+2G3Pw7LT87PbHdbYQOnSOmOn5XDyll35BHdNaGsnhzcAQsL/fxCvLmZMBzLtZ7uxOJuOd3gtbKXqKxe8tFz2LHMc34Km5FAixrrMDoqKPErOn0N3vtf4uHtWmvS9xi4BGkG5L5JZbY2IU1e0ngLFzeLtpyenjacIxEpszsFClRjpq2HUzDxvsPDFKxOUdKS3kuYv32qQuzuwn434PxeVcLBtxrL7AXIbC8yvLexLcos6KOwJoP/qz9lAuwcln+kwG2tdNup+4vuFXom+g3NPodhW5t5ldNtsi6wDv8ok8OP1eIm/3fKClRpM6Z/HifNDjgd4mU03F0dPYy/Qpp38Fig9nSmuN8eIE/fvRsK1fNClEuyvEN7SQ0eE4Wc90GoyW7t5/sTNacF+5AyosY+sLsm42aACfdY8UHqGjtw1PzN4f+gEEaW+MfI0pRNBWarjBDTtuTDryjO4rX8H6otYhU=", "tests/gave2_ensemble/test_training_v2.py": "eNq9WG2P4kYS/s6vsPhkS+AYM8zujISUKMqdToqS6LR3XxBqNaYMnWm7fd1tZtko//2q2m8YDMxmd4N2B9z12lVPV1VbZIXS1hPuS4pNWFohR9WjZ46m+WkhK1IhoXkuc2EtGDtKtcq8gts9CtdqvN/wcdRw5mVWHD1uvLwYjUb//vXXD97ScfiMkUbGglCDUfIAfhAWXENuzSpej0TqGat9kgi8XKGTOXkUkrHnkYef5ikUuQFt/WjSSQRoLJHcGO+D5iIX+e6/8Qd02PiN6yE9/sgNBJW2LaQerbNc5cAKDYVWCRiDouwTaAWGwQH0kSV7nucgmSqtEVtgWgnfgExrPfRxUYGPBWiR0X7CHT9AzAD9zDYSwp72JmzdIsvUlkthBZhRqzNJC4xcXoRpKaXvz+KJN3ucePNg4kXhYuJt7bGAJdGl4nYeB503KWd8SHbmZOd3ZQ/Xhd/dEsbIVJIUP9MXvSm1ip+nyDt/ns7XqGEWRl0cMOxFaXF1MF4+hmlCKiaV6/XXYeIhDyzHlNxx0GmjxIUIE8TPT/8rufQr9aHZ8wImXuvyIgiuy7gt1JIrRGG0Dk2Z+YGLUDQo+E8N3ILui2IWF2eifWgWyuAuD8BeQez21jBKyZFtuOR5gkjE08NSpWGnVZlvGcf/iRTF58LT1keGUTUw7BC3EL2wj5pYgsbsCVJrIqXoBr/f8tOn4SREcK350V89hBjKOKK/CIB1MOkJWGW5POGeRRXnyde5SIabyspsSYr7BP7REZyxlnKCEjRD8ceQ1OljXMpEKgN+vbOJ17lS+dy6j2542iq5nMH08Tyje2Gs2mmeIUZB852rPIkwQuVsCxYSDJrBIie4xPTmL+jCV8vmpUWXntalk5Q26ekfZ9zkxUl+fOjw7nL0eSKNITpG1dmP+uqq84WEpwFCI9FSMrBaJATFu5v1G9OTSt9wkfhBZsrUx75Wvhpf6q57hBmvV9Ha4Td4szbUcuAS6ECfqYnCWfAFTkmR2gvPBqqMBK4dYHZYopiG3x0K1QZNHQArisqN5TlWIy2UZmaPWEpK+/VgiXvCmt7z4qT8125gSv/oHeIxFCrZj59xS/3TPTYK970VCSARt/v+YfH+8Wk+f/8wjxZxPD/jJkZmowUyz8PZ0/zh8WkRz97NH9/NFzCNFmfsG4pYa/rcMuIIKqpB8lNH/rNrd+B2vRzct79q9ou5qutXrW85i4Yh+g8usS5Valdj9TJeD4Lmgy7B5/nRb5DitUgZ06yFDcqo3MN20vx0i5XWagVBFNwGD08SKBA8CbbnjcYyhjAU+RYKwD8IoW9W2m5iyC3fhNB5li8Ok4NSHN9jc2fOIWnxJtbueD57qzikcknzWRy+X98G9dN1FEfn7W6Mzet0TGi1nJ6cP0fXQTMI1DqoAzhdBA0M+1Cxew1UX5JSGxoSMBToCmHEsD3GB0kHAa9YcvAc7doh4quBpLPc0/+mtn9F1qdxfIXIqAdrnMzXXfd/d7d60+x3TfWChsLZZbl2+6M5DweGlJcSjxvNgfKBGZ7CZ0ariU2l9BCfFF4tdiJ3/ZwuXlzvDqvnddeD9fG5h7KGCflXPYKDYKM/LI7jySV5Ot1yy6daKTtEJuKwmC7z6VboISKSzHfVbpNM69ccLDowrCZVcjvNeC5SjPDblX1HcqyRC3/HGjms33LzMkSh9RsuDVGis8V17wkzQNNPF26uDZZlXPVPLnmUV3k9f03qb1ybSGO4wdt0W8Ho3hTcEcixSmhA8+CwiIUuuGvCJntmxCega+QdZpzrttSCyuwNmnmGF8fxJp09ju+xVoUNHYije6xSo2mYPtznY9QKaeojvXc94BrvfTi6FgVVtk50ftej7tqM262vwwMS/zK/IK0SwfsRe6WmgZW9xJpzXoEczWI9wR6WCcsENvw9JC9YuXk9HLENtRqu6Yr+V+r3SdnurMHHPS+Nhe2NOWiA28e6qq1zblnfF893uKTN451OvRIHzcjBVzcR0ypZiGdP4VNwfUj7Yv1xtQFn4Pv2/Zd5EcV/colA8Pvv/0I8kNhzC0j8sVUaJ6LAE8a9g6vCUq02a4KuA1LCtgZSC4v6tcSWYcc0DEc+HAI/AfZ2gaMPKjiyg1DSDYUXoGjaEBkavRkqZAhMI3zArYE8MUfkThlesgTNgM4GNvrcKO2v8DPFq/sa2/Zq1f6oltYnk7TIr8qfSbWKTsUzLPStLAbV0NW4/jc81v9MiRrcku88mTidOCIM89T+1lx/PxhShXcAZnHqNS9HLPqvXG/bVxtYyxKlcfCxp0Pgt8BE3w0HzFdh9/hrJ05HPzxeO7BD2Z3RsaLXOm7Qq3/fzi2WxRfwK5UdW73jIRsL0jtdVEam3VPPTHMPbxW4dyyVrVr3DST1ev3tqDTaJnVUGqR9norK2as66IW9SD3Gco6FjnnLpTdmOEph6WfjCgMtWGkVx5f/AxkPs38="}
patched_files = []
for relative, payload in EMBEDDED_SOURCE_PATCHES.items():
    destination = WORK_ROOT / relative
    corrected = zlib.decompress(base64.b64decode(payload))
    if not destination.is_file() or destination.read_bytes() != corrected:
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(corrected)
        patched_files.append(relative)
print({"corrected_source_files": patched_files, "count": len(patched_files)})

In [ ]:
requirements = WORK_ROOT / "experiments" / "gave2_ensemble" / "requirements-gave2-main.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
import torch
assert torch.cuda.is_available(), "Select a Colab GPU runtime"
assert torch.cuda.is_bf16_supported(), "The selected GPU must support BF16"
gpu = torch.cuda.get_device_properties(0)
print({"torch": torch.__version__, "gpu": gpu.name, "vram_gib": round(gpu.total_memory / 1024**3, 2)})

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
OOF_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
run_module(
    "experiments.gave2_ensemble.integrity_v2",
    "--data-root", DATA_ROOT,
    "--run-dir", RUN_DIR,
    "--seed", SEED,
    "--folds", N_FOLDS,
)
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests/gave2_ensemble", "-p", "test_*.py"],
    cwd=WORK_ROOT,
    check=True,
)
assert FOLD_MANIFEST.is_file()
print("Dataset, folds, and packaged tests passed.")

In [ ]:
MEMORY_TEST_PATH = WORK_ROOT / "experiments" / "gave2_ensemble" / "memory_test_v2.py"
ORIGINAL_MEMORY_TEST = MEMORY_TEST_PATH.read_text(encoding="utf-8")

def benchmark_profile(task, base_channels, batch_size, activation_checkpointing, steps=5):
    source = ORIGINAL_MEMORY_TEST
    replacements = {
        "train_ids[: max(2, args.steps)]": f"train_ids[: max(2, args.steps * {batch_size})]",
        "batch_size=1, shuffle=False": f"batch_size={batch_size}, shuffle=False",
        "activation_checkpointing=True,": f"activation_checkpointing={activation_checkpointing},",
    }
    for old, new in replacements.items():
        if source.count(old) != 1:
            raise RuntimeError(f"Memory-test compatibility patch failed for: {old}")
        source = source.replace(old, new)
    MEMORY_TEST_PATH.write_text(source, encoding="utf-8")
    for bytecode in (MEMORY_TEST_PATH.parent / "__pycache__").glob("memory_test_v2*.pyc"):
        bytecode.unlink()
    started = time.perf_counter()
    try:
        run_module(
            "experiments.gave2_ensemble.memory_test_v2",
            "--data-root", DATA_ROOT,
            "--fold-manifest", FOLD_MANIFEST,
            "--task", task,
            "--base-channels", base_channels,
            "--num-refinements", NUM_REFINEMENTS,
            "--steps", steps,
        )
    except subprocess.CalledProcessError:
        return None
    finally:
        MEMORY_TEST_PATH.write_text(ORIGINAL_MEMORY_TEST, encoding="utf-8")
        for bytecode in (MEMORY_TEST_PATH.parent / "__pycache__").glob("memory_test_v2*.pyc"):
            bytecode.unlink()
    elapsed = time.perf_counter() - started
    benchmark_images_per_second = steps * batch_size / elapsed
    return {
        "base_channels": base_channels,
        "batch_size": batch_size,
        "grad_accum": 1,
        "activation_checkpointing": activation_checkpointing,
        "benchmark_seconds": elapsed,
        "benchmark_images_per_second": benchmark_images_per_second,
    }

def select_largest_safe_batch_profile(task):
    base_channels = BASE_CHANNELS[task]
    attempted = []
    for batch_size in PREFERRED_BATCH_SIZES:
        print(f"Testing {task}: base={base_channels}, batch={batch_size}, checkpointing=False")
        profile = benchmark_profile(task, base_channels, batch_size, False)
        if profile is not None:
            return profile
        attempted.append(batch_size)
        print(f"{task} batch={batch_size} failed the memory gate; trying a smaller batch.")
    raise RuntimeError(
        f"No safe batch profile for {task} base={base_channels}; attempted batches {attempted}"
    )

TRAINING_PROFILES = {
    "task2": select_largest_safe_batch_profile("task2"),
    "task1": select_largest_safe_batch_profile("task1"),
}
(RUN_DIR / "selected_memory_profile.json").write_text(json.dumps(TRAINING_PROFILES, indent=2))
print("Selected largest safe batch profiles:", json.dumps(TRAINING_PROFILES, indent=2))

In [ ]:
def train_fold(task, fold, epochs, max_wall_minutes):
    fold_dir = RUN_DIR / "cmrrwnet_v2" / task / f"fold_{fold}"
    profile = TRAINING_PROFILES[task]
    arguments = [
        "--data-root", DATA_ROOT,
        "--run-dir", RUN_DIR,
        "--fold-manifest", FOLD_MANIFEST,
        "--task", task,
        "--fold", fold,
        "--base-channels", profile["base_channels"],
        "--num-refinements", NUM_REFINEMENTS,
        "--batch-size", profile["batch_size"],
        "--grad-accum", 1,
        "--workers", 2,
        "--epochs", epochs,
        "--max-wall-minutes", max_wall_minutes,
        "--amp", "bf16",
        "--lr", LEARNING_RATE,
        "--seed", SEED,
    ]
    if not profile["activation_checkpointing"]:
        arguments.append("--no-activation-checkpointing")
    if (fold_dir / "last.pt").is_file():
        arguments.append("--resume")
    run_module("experiments.gave2_ensemble.train_v2", *arguments)

train_fold(
    "task2",
    0,
    LEARNING_GATE_EPOCHS,
    bounded_training_minutes(TASK2_TRAINING_MINUTES * 0.4),
)
history_path = RUN_DIR / "cmrrwnet_v2" / "task2" / "fold_0" / "history.json"
history = json.loads(history_path.read_text())
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
from experiments.gave2_ensemble.training_utils_v2 import assess_learning_gate

gate_report = assess_learning_gate(history, minimum_epochs=LEARNING_GATE_EPOCHS)
print("Learning gate:", json.dumps(gate_report, indent=2))
assert gate_report["ok"], "Task 2 fold 0 failed spatial learning: " + "; ".join(gate_report["reasons"])
print("Learning gate passed:", history[-1])

In [ ]:
train_fold(
    "task2",
    0,
    TASK2_MAX_EPOCHS,
    bounded_training_minutes(TASK2_TRAINING_MINUTES * 0.1),
)
for fold in range(1, N_FOLDS):
    train_fold(
        "task2",
        fold,
        TASK2_MAX_EPOCHS,
        bounded_training_minutes(TASK2_TRAINING_MINUTES * 0.5),
    )
print("All Task 2 folds finished or resumed to completion.")

In [ ]:
for fold in range(N_FOLDS):
    train_fold(
        "task1",
        fold,
        TASK1_MAX_EPOCHS,
        bounded_training_minutes(TASK1_TRAINING_MINUTES / N_FOLDS),
    )
print("All Task 1 folds finished or resumed to completion.")

In [ ]:
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
from experiments.gave2_ensemble.integrity_v2 import certified_checkpoints

certification = {}
for task in ("task1", "task2"):
    checkpoints = certified_checkpoints(RUN_DIR, task, expected_folds=N_FOLDS)
    best_scores = []
    for checkpoint in checkpoints:
        history = json.loads(checkpoint.with_name("history.json").read_text())
        best_scores.append(max(row["soft_dice"] for row in history))
    certification[task] = {"checkpoints": [str(path) for path in checkpoints], "best_soft_dice": best_scores}
(RUN_DIR / "fold_certification.json").write_text(json.dumps(certification, indent=2))
print(json.dumps(certification, indent=2))

In [ ]:
selection = {}
for task in ("task2", "task1"):
    raw_root = OOF_ROOT / "raw_none"
    run_module(
        "experiments.gave2_ensemble.predict_v2",
        "--data-root", DATA_ROOT,
        "--run-dir", RUN_DIR,
        "--fold-manifest", FOLD_MANIFEST,
        "--output-root", raw_root,
        "--team-id", TEAM_ID,
        "--task", task,
        "--mode", "oof",
        "--expected-folds", N_FOLDS,
        "--tta", "none",
    )
    task_name = "Task1" if task == "task1" else "Task2"
    probability_dir = raw_root / "cmrrwnet_v2" / TEAM_ID / task_name
    calibration_path = RUN_DIR / "calibration" / f"{task}_none.json"
    calibrated_dir = OOF_ROOT / "calibrated" / f"{task}_none"
    run_module(
        "experiments.gave2_ensemble.probability_calibration_v2",
        "--data-root", DATA_ROOT,
        "--probability-dir", probability_dir,
        "--task", task,
        "--output", calibration_path,
        "--calibrated-dir", calibrated_dir,
    )
    report = json.loads(calibration_path.read_text())
    score = report["calibrated_dice_mean"]
    if score <= 0.25 or min(report["calibrated_dice_channels"]) <= 0.15:
        print("OOF quality warning:", json.dumps(report, indent=2))
    selection[task] = {
        "tta": "none",
        "calibration": str(calibration_path),
        "calibrated_oof_dir": str(calibrated_dir),
        "report": report,
    }
(RUN_DIR / "oof_selection.json").write_text(json.dumps(selection, indent=2))
print(json.dumps(selection, indent=2))

In [ ]:
selection = json.loads((RUN_DIR / "oof_selection.json").read_text())
calibrator_path = RUN_DIR / "task3_calibrator.json"
run_module(
    "experiments.gave2_ensemble.biomarkers_v2", "fit",
    "--data-root", DATA_ROOT,
    "--oof-task2-dir", selection["task2"]["calibrated_oof_dir"],
    "--output", calibrator_path,
)
task3_calibrator = json.loads(calibrator_path.read_text())
print({key: value["accepted"] for key, value in task3_calibrator["targets"].items()})

In [ ]:
for task in ("task2", "task1"):
    chosen = selection[task]
    run_module(
        "experiments.gave2_ensemble.predict_v2",
        "--data-root", DATA_ROOT,
        "--run-dir", RUN_DIR,
        "--fold-manifest", FOLD_MANIFEST,
        "--output-root", OUTPUT_ROOT,
        "--team-id", TEAM_ID,
        "--task", task,
        "--mode", "validation",
        "--expected-folds", N_FOLDS,
        "--tta", "none",
        "--calibration", chosen["calibration"],
        "--accumulator-dir", WORK_ROOT / ".prediction_accumulator",
    )
TEAM_ROOT = OUTPUT_ROOT / "cmrrwnet_v2" / TEAM_ID
print("Task 1 and Task 2 predictions:", TEAM_ROOT)

In [ ]:
run_module(
    "experiments.gave2_ensemble.biomarkers_v2", "predict",
    "--data-root", DATA_ROOT,
    "--task2-dir", TEAM_ROOT / "Task2",
    "--output-dir", TEAM_ROOT / "Task3",
    "--calibrator", calibrator_path,
    "--split", "validation",
)
assert len(list((TEAM_ROOT / "Task3").glob("*.txt"))) == 50
print("Task 3 predictions complete.")

In [ ]:
report_path = RUN_DIR / "final_submission_report.json"
run_module(
    "experiments.gave2_ensemble.submission_v2",
    "--data-root", DATA_ROOT,
    "--team-root", TEAM_ROOT,
    "--output-zip", FINAL_ZIP,
    "--report", report_path,
)
report = json.loads(report_path.read_text())
assert report["ok"] and report["counts"] == {"Task1": 50, "Task2": 50, "Task3": 50}
PACKAGING_SUCCEEDED = True
print(json.dumps(report, indent=2, ensure_ascii=False))

In [ ]:
with zipfile.ZipFile(FINAL_ZIP) as archive:
    assert archive.testzip() is None
    names = archive.namelist()
    assert len(names) == 150
    assert all(name.startswith(f"{TEAM_ID}/") for name in names)
    assert len({name.split("/")[1] for name in names}) == 3
final_sha256 = sha256_file(FINAL_ZIP)
ZIP_READBACK_SUCCEEDED = True
print({"ready_to_submit": str(FINAL_ZIP), "sha256": final_sha256, "payload_files": len(names)})

In [ ]:
assert PACKAGING_SUCCEEDED, "Current run did not complete submission packaging"
assert ZIP_READBACK_SUCCEEDED, "Current run did not complete ZIP integrity readback"
if AUTO_DISCONNECT:
    print("Submission verified. Disconnecting the Colab runtime in 10 seconds.", flush=True)
    time.sleep(10)
    from google.colab import runtime

    runtime.unassign()
else:
    print("Submission verified. AUTO_DISCONNECT is disabled.", flush=True)

## Result

Submit the `FINAL_ZIP` printed above only after the final cell reports 150 payload files. Keep the V5 run directory, OOF selection report, and submission report in Drive for reproducibility.